In [7]:
with open('wiki_clean.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [8]:
text[:1000]

'<SOA>\nTITLE: Lego Star Wars (disambiguation)\n\nABSTRACT:\nLego Star Wars is a Lego theme based around models of vehicles and sets from the Star Wars franchise. It can also refer to:\n  - Lego Star Wars: The Video Game, a video game that provides a humorous and child-oriented retelling of the Star Wars prequel trilogy\n    - Lego Star Wars II: The Original Trilogy, the sequel to Lego Star Wars: The Video Game that retells the original trilogy\n    - Lego Star Wars: The Complete Saga, a compilation of the first two games\n    - Lego Star Wars III: The Clone Wars, the third in the series of games\n    - Lego Star Wars: The Force Awakens\n    - Lego Star Wars: The Skywalker Saga, an updated version following the story of all 9 feature films\n  - Several animated shorts and specials airing on Cartoon Network\n    - Lego Star Wars: Revenge of the Brick, a 2005 mini-movie based on Revenge of the Sith\n    - Lego Star Wars: The Quest for R2-D2, a 2009 animated mini-movie and online game bas

In [9]:
print("len: ", len(text))

len:  1041830


In [10]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)

	
 !"#$%&'()*+,-./0123456789:;<=>?ABCDEFGHIJKLMNOPQRSTUVWXYZ[\]_`abcdefghijklmnopqrstuvwxyz{|}~
95


### tokenizer

In [11]:
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode("hello world"))
print(decode(encode("hello world")))

[72, 69, 76, 76, 79, 2, 87, 79, 82, 76, 68]
hello world


In [12]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100])

torch.Size([1041830]) torch.int64
tensor([30, 52, 48, 34, 32,  1, 53, 42, 53, 45, 38, 28,  2, 45, 69, 71, 79,  2,
        52, 84, 65, 82,  2, 56, 65, 82, 83,  2, 10, 68, 73, 83, 65, 77, 66, 73,
        71, 85, 65, 84, 73, 79, 78, 11,  1,  1, 34, 35, 52, 53, 51, 34, 36, 53,
        28,  1, 45, 69, 71, 79,  2, 52, 84, 65, 82,  2, 56, 65, 82, 83,  2, 73,
        83,  2, 65,  2, 45, 69, 71, 79,  2, 84, 72, 69, 77, 69,  2, 66, 65, 83,
        69, 68,  2, 65, 82, 79, 85, 78, 68,  2])


### split dataset

In [49]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [50]:
block_size = 8
train_data[:block_size+1]

tensor([30, 52, 48, 34, 32,  1, 53, 42, 53])

In [51]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
print(x)
print(y)

tensor([30, 52, 48, 34, 32,  1, 53, 42])
tensor([52, 48, 34, 32,  1, 53, 42, 53])


In [52]:
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"input: {context}, target: {target}")

input: tensor([30]), target: 52
input: tensor([30, 52]), target: 48
input: tensor([30, 52, 48]), target: 34
input: tensor([30, 52, 48, 34]), target: 32
input: tensor([30, 52, 48, 34, 32]), target: 1
input: tensor([30, 52, 48, 34, 32,  1]), target: 53
input: tensor([30, 52, 48, 34, 32,  1, 53]), target: 42
input: tensor([30, 52, 48, 34, 32,  1, 53, 42]), target: 53


In [53]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size):
    for t in range(block_size):     # time dim
        context = x[:t+1]
        target = y[t]
        print(f"input: {context}, target: {target}")

inputs:
torch.Size([4, 8])
tensor([[83, 69,  2, 68, 85, 82, 73, 78],
        [16,  2, 36, 85, 82, 82, 69, 78],
        [ 2, 73, 83,  2, 79, 78, 69,  2],
        [65, 78, 84, 65,  2, 66, 76, 85]])
targets:
torch.Size([4, 8])
tensor([[69,  2, 68, 85, 82, 73, 78, 71],
        [ 2, 36, 85, 82, 82, 69, 78, 84],
        [73, 83,  2, 79, 78, 69,  2, 79],
        [78, 84, 65,  2, 66, 76, 85, 69]])
----
input: tensor([30]), target: 52
input: tensor([30, 52]), target: 48
input: tensor([30, 52, 48]), target: 34
input: tensor([30, 52, 48, 34]), target: 32
input: tensor([30, 52, 48, 34, 32]), target: 1
input: tensor([30, 52, 48, 34, 32,  1]), target: 53
input: tensor([30, 52, 48, 34, 32,  1, 53]), target: 42
input: tensor([30, 52, 48, 34, 32,  1, 53, 42]), target: 53
input: tensor([30]), target: 52
input: tensor([30, 52]), target: 48
input: tensor([30, 52, 48]), target: 34
input: tensor([30, 52, 48, 34]), target: 32
input: tensor([30, 52, 48, 34, 32]), target: 1
input: tensor([30, 52, 48, 34, 32,  

### bigrams

In [54]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLangModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)    # (B, T, C)     (batch_size, context_size, vocab_size)

        if targets == None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            # only last time step
            logits = logits[:, -1, :]   # (B, C)
            # get probs
            probs = F.softmax(logits, dim=-1)   # (B, C)
            # sample
            idx_next = torch.multinomial(probs, num_samples=1)  # (B, 1)
            idx = torch.cat((idx, idx_next), dim=1)     # (B, T+1)
        return idx


m = BigramLangModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

torch.Size([32, 95])
tensor(5.2641, grad_fn=<NllLossBackward0>)


In [55]:
idx = torch.zeros((1, 1), dtype=torch.long)
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

	[:>&5!KF!u]c(>A12'5_c&4d?7#|N[c2Vh!nTa$M'56wHaK;3k{Hk;1nmKi|c1|OOgxa4!5iU,E6E5VbjqR0P|B|m&Q1+(Nd`aWy


^ garbage because I didn't train

In [56]:
# optimizer
optimizer = torch.optim.Adam(m.parameters(), lr=1e-3)

In [57]:
batch_size = 32

for steps in range(10000):
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

2.5575051307678223


In [58]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=300)[0].tolist()))

	6>xutamajo Ugr illelio ioso ise A>
###201400019, muerid qu. In cicatiethiof e bsctoful, caul in ticatoon a ths theiee vewsean olans s Aqupo Preshilanduan, acche op Ponfty mo&Thath d feddie asthe g Col fon: SIQacomenofraz}. J3. on Gonn tof Fo arimometedeve Ade owemabed gnbole: f rt theand acirynde ac


^ less garbage